# Counterfactual Evaluation — Before vs After Fine-tuning

Metrics measuring how much the fine-tuning improved counterfactual explanation quality.

| | File | Field |
|--|------|-------|
| **Before** | `counterfactual_training_dataset.json` | `llama_explanation` |
| **After**  | `counterfactual_finetuned_results.json` | `llama_explanation` |
| **Ref**    | `counterfactual_finetuned_results.json` | `chatgpt_reference` |

## Metrics
| Metric | What it checks |
|--------|----------------|
| Basic Stats | avg length, template fallback rate |
| BMR | does the explanation name the bridge items? |
| USR | are explanations diverse (not repeated)? |
| Alpha-Sentiment | do harder items (higher α) get appropriately hedged language? |
| Factual Precision | fraction of reference content-words found in the explanation |
| BARTScore | log P(explanation \| reference) via BART-large-CNN |

In [3]:
# Cell 1: Install dependencies
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'evaluate', 'bert_score', 'scipy', 'pandas',
                'transformers', 'torch', 'sentencepiece'],
               check=True)
print('OK')

OK


In [4]:
# Cell 2: Load data
import json, numpy as np, pandas as pd
from scipy import stats

TRAINING_JSON  = 'counterfactual_training_dataset.json'
FINETUNED_JSON = 'counterfactual_finetuned_results.json'

with open(TRAINING_JSON) as f:  train_data = json.load(f)
with open(FINETUNED_JSON) as f: ft_data    = json.load(f)

train_by_key = {(r['user_id'], r.get('target_name', '')): r for r in train_data}

records = []
for r in ft_data:
    uid        = r['user_id']
    target     = r.get('target_name', '')
    after_exp  = r.get('llama_explanation', '').strip()
    ref        = r.get('chatgpt_reference', '').strip()
    if not after_exp or not ref:
        continue
    orig = train_by_key.get((uid, target))
    before_exp = orig.get('llama_explanation', '').strip() if orig else ''
    if not before_exp:
        continue
    records.append({
        'user_id':       uid,
        'target_name':   target,
        'before_exp':    before_exp,
        'after_exp':     after_exp,
        'ref':           ref,
        'bridge_names':  r.get('bridge_names', []),
        'minimum_alpha': r.get('minimum_alpha', float('nan')),
    })

df = pd.DataFrame(records)
print(f'Training records : {len(train_data)}')
print(f'Finetuned records: {len(ft_data)}')
print(f'Aligned pairs    : {len(df)}')
print()
print('Sample:')
print(f'  BEFORE : {df["before_exp"].iloc[0][:120]}')
print(f'  AFTER  : {df["after_exp"].iloc[0][:120]}')
print(f'  REF    : {df["ref"].iloc[0][:120]}')

Training records : 5668
Finetuned records: 2200
Aligned pairs    : 2199

Sample:
  BEFORE : the user is drawn to books that delve into human nature and provide insightful observations, but they seem uninterested 
  AFTER  : Your profile suggests that Collapse is not strongly supported as a recommendation. Your profile emphasizes Based on the 
  REF    : Based on the patterns in your profile, the system would not recommend Collapse. Your profile emphasizes Based on the use


---
## Section 1: Basic Statistics

In [5]:
TEMPLATE_SIGNAL = "is not currently in the user's top"

df['before_len']      = df['before_exp'].apply(lambda x: len(x.split()))
df['after_len']       = df['after_exp'].apply(lambda x: len(x.split()))
df['before_template'] = df['before_exp'].apply(lambda x: TEMPLATE_SIGNAL in x and 'alpha=' in x)
df['after_template']  = df['after_exp'].apply(lambda x: TEMPLATE_SIGNAL in x and 'alpha=' in x)

print('=' * 62)
print('  BASIC STATISTICS')
print('=' * 62)
print(f'  {"Metric":<30} {"Before":>12} {"After":>12}')
print('  ' + '-' * 56)
print(f'  {"Avg length (words)":<30} {df["before_len"].mean():>12.1f} {df["after_len"].mean():>12.1f}')
print(f'  {"Median length (words)":<30} {df["before_len"].median():>12.1f} {df["after_len"].median():>12.1f}')
print(f'  {"Template fallback rate":<30} {df["before_template"].mean()*100:>11.1f}% {df["after_template"].mean()*100:>11.1f}%')
print('=' * 62)

  BASIC STATISTICS
  Metric                               Before        After
  --------------------------------------------------------
  Avg length (words)                     33.4        155.6
  Median length (words)                  31.0        156.0
  Template fallback rate                 0.0%         0.0%


---
## Section 2: Bridge Mention Rate (BMR)

In [6]:
def mentions_bridge(explanation, bridge_names):
    if not isinstance(bridge_names, list): return False
    exp_lower = explanation.lower()
    for name in bridge_names:
        keywords = [w.lower() for w in str(name).split() if len(w) > 3]
        if any(kw in exp_lower for kw in keywords):
            return True
    return False

df['before_bmr'] = df.apply(lambda r: mentions_bridge(r['before_exp'], r['bridge_names']), axis=1)
df['after_bmr']  = df.apply(lambda r: mentions_bridge(r['after_exp'],  r['bridge_names']), axis=1)

b_bmr = df['before_bmr'].mean()
a_bmr = df['after_bmr'].mean()

print('=' * 62)
print('  BRIDGE MENTION RATE (BMR)')
print('=' * 62)
print(f'  {"Metric":<30} {"Before":>12} {"After":>12} {"Delta":>6}')
print('  ' + '-' * 62)
print(f'  {"BMR (any bridge)":<30} {b_bmr:>11.4f} {a_bmr:>12.4f} {a_bmr-b_bmr:>+6.4f}')
print('=' * 62)

  BRIDGE MENTION RATE (BMR)
  Metric                               Before        After  Delta
  --------------------------------------------------------------
  BMR (any bridge)                    0.4402       0.9332 +0.4930


---
## Section 3: USR — Unique Sentence Rate

In [7]:
def usr(texts):
    n = [t.lower().strip() for t in texts]
    return len(set(n)) / len(n)

b_usr = usr(df['before_exp'].tolist())
a_usr = usr(df['after_exp'].tolist())

print('=' * 62)
print('  UNIQUE SENTENCE RATE (USR)')
print('=' * 62)
print(f'  {"Metric":<30} {"Before":>12} {"After":>12} {"Delta":>6}')
print('  ' + '-' * 62)
print(f'  {"USR":<30} {b_usr:>11.4f} {a_usr:>12.4f} {a_usr-b_usr:>+6.4f}')
print('=' * 62)

  UNIQUE SENTENCE RATE (USR)
  Metric                               Before        After  Delta
  --------------------------------------------------------------
  USR                                 0.9845       1.0000 +0.0155


---
## Section 4: Alpha-Sentiment Correlation

In [8]:
POSITIVE_WORDS = ['enjoy','love','great','excellent','perfect','match','align',
                  'relevant','interested','appreciate','recommend','captivating',
                  'engaging','compelling','fascinating','would enjoy','will enjoy']
NEGATIVE_WORDS = ['unlikely','not recommended','may not','might not','unsure',
                  'gap','distant','unrelated','missed','failed','not in']

def sentiment(text):
    t = text.lower()
    return sum(1 for w in POSITIVE_WORDS if w in t) - sum(1 for w in NEGATIVE_WORDS if w in t)

valid = df.dropna(subset=['minimum_alpha'])
alphas = valid['minimum_alpha'].values
b_sent = valid['before_exp'].apply(sentiment).values
a_sent = valid['after_exp'].apply(sentiment).values

b_r, b_p = stats.pearsonr(alphas, b_sent)
a_r, a_p = stats.pearsonr(alphas, a_sent)

print('=' * 62)
print('  ALPHA-SENTIMENT CORRELATION (Pearson r)')
print('=' * 62)
print(f'  {"Metric":<30} {"Before":>12} {"After":>12} {"Delta":>6}')
print('  ' + '-' * 62)
print(f'  {"Pearson r":<30} {b_r:>12.4f} {a_r:>12.4f} {a_r-b_r:>+6.4f}')
print(f'  {"p-value":<30} {b_p:>12.4f} {a_p:>12.4f}')
print('=' * 62)

  ALPHA-SENTIMENT CORRELATION (Pearson r)
  Metric                               Before        After  Delta
  --------------------------------------------------------------
  Pearson r                           -0.0015       0.0118 +0.0133
  p-value                              0.9438       0.5799


---
## Section 5: Factual Precision

Measures what fraction of **content words** from the reference appear in the explanation.
Stop words and short words (≤3 chars) are excluded so we focus on meaningful terms.

- **Higher is better** — the explanation covers more key facts from the reference.

In [9]:
STOP_WORDS = {
    'the','a','an','is','are','was','were','be','been','being',
    'have','has','had','do','does','did','will','would','could',
    'should','may','might','to','of','in','for','on','with',
    'at','by','from','that','this','it','its','as','and','or',
    'but','not','no','so','if','than','then','also','which','very',
    'more','some','such','into','about','over','after','just','how',
    'they','their','them','user','item','product','because','when',
}

def content_words(text):
    return {w.lower().strip('.,!?;:\'"()[]')
            for w in text.split()
            if len(w) > 3 and w.lower() not in STOP_WORDS}

def factual_precision(hypothesis, reference):
    """Fraction of reference content words found in hypothesis."""
    ref_words = content_words(reference)
    hyp_words = content_words(hypothesis)
    if not ref_words:
        return 0.0
    return len(ref_words & hyp_words) / len(ref_words)

df['before_fp'] = df.apply(lambda r: factual_precision(r['before_exp'], r['ref']), axis=1)
df['after_fp']  = df.apply(lambda r: factual_precision(r['after_exp'],  r['ref']), axis=1)

b_fp = df['before_fp'].mean()
a_fp = df['after_fp'].mean()

print('=' * 62)
print('  FACTUAL PRECISION')
print('=' * 62)
print(f'  {"Metric":<30} {"Before":>12} {"After":>12} {"Delta":>6}')
print('  ' + '-' * 62)
print(f'  {"Factual Precision":<30} {b_fp:>11.4f} {a_fp:>12.4f} {a_fp-b_fp:>+6.4f}')
print('=' * 62)

  FACTUAL PRECISION
  Metric                               Before        After  Delta
  --------------------------------------------------------------
  Factual Precision                   0.0832       0.4958 +0.4126


---
## Section 6: BARTScore (Bidirectional, facebook/bart-large)

BARTScore measures **semantic equivalence** between the generated explanation and the ChatGPT reference.


We average two directions:
- `P(explanation | reference)` — can BART "reconstruct" our explanation from the reference?
- `P(reference | explanation)` — can BART reconstruct the reference from our explanation?
Averaging both directions is more robust than one-way scoring for **paraphrase-style evaluation**.


In [10]:
import torch
from transformers import BartForConditionalGeneration, BartTokenizer


BART_MODEL = 'facebook/bart-large'

print(f'Loading {BART_MODEL}')
bart_tokenizer = BartTokenizer.from_pretrained(BART_MODEL)
bart_model     = BartForConditionalGeneration.from_pretrained(BART_MODEL)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
bart_model = bart_model.to(device)
bart_model.eval()
print(f'BART loaded on {device}')


def _bart_one_direction(srcs, hyps, batch_size=4, max_len=256):
    """Compute mean log P(hyp | src) per non-pad token."""
    scores = []
    for i in range(0, len(srcs), batch_size):
        b_srcs = srcs[i:i+batch_size]
        b_hyps = hyps[i:i+batch_size]

        src_enc = bart_tokenizer(
            b_srcs, return_tensors='pt', padding=True,
            truncation=True, max_length=max_len
        ).to(device)
        tgt_enc = bart_tokenizer(
            b_hyps, return_tensors='pt', padding=True,
            truncation=True, max_length=max_len
        )
        labels = tgt_enc['input_ids'].to(device)

        with torch.no_grad():
            out      = bart_model(**src_enc, labels=labels)
            logits   = out.logits                                  # (B, T, V)
            log_probs = torch.nn.functional.log_softmax(logits, dim=-1)
            target   = labels[:, 1:]                               # drop BOS
            lp       = log_probs[:, :-1, :]                        # align
            gathered = lp.gather(2, target.clamp(min=0).unsqueeze(2)).squeeze(2)
            mask     = (target != bart_tokenizer.pad_token_id).float()
            per_sample = (gathered * mask).sum(1) / mask.sum(1).clamp(min=1)
            scores.extend(per_sample.cpu().tolist())

        if (i // batch_size) % 10 == 0:
            print(f'  {i+len(b_srcs)}/{len(srcs)}', end='\r')
    print()
    return scores


def bartscore_bidirectional(refs, hyps, batch_size=4, max_len=256):
    """
    Symmetric BARTScore = average of:
      P(hyp | ref)  — can BART generate our explanation from the reference?
      P(ref | hyp)  — can BART regenerate the reference from our explanation?
    Averaging both directions for paraphrase-style evaluation
    (it penalises both missing content AND hallucinated content).
    """
    forward  = _bart_one_direction(refs, hyps, batch_size, max_len)   # P(hyp|ref)
    backward = _bart_one_direction(hyps, refs, batch_size, max_len)   # P(ref|hyp)
    return [(f + b) / 2 for f, b in zip(forward, backward)]


refs_list   = df['ref'].tolist()
before_list = df['before_exp'].tolist()
after_list  = df['after_exp'].tolist()

print('\nBidirectional BARTScore — BEFORE explanations...')
b_bart_scores = bartscore_bidirectional(refs_list, before_list)
print('Bidirectional BARTScore — AFTER explanations...')
a_bart_scores = bartscore_bidirectional(refs_list, after_list)

df['before_bart'] = b_bart_scores
df['after_bart']  = a_bart_scores

b_bart_mean = np.mean(b_bart_scores)
a_bart_mean = np.mean(a_bart_scores)
b_bart_std  = np.std(b_bart_scores)
a_bart_std  = np.std(a_bart_scores)

print('=' * 68)
print(f'  BARTSCORE (bidirectional, {BART_MODEL})')
print('  log [(P(exp|ref) + P(ref|exp)) / 2] per token — higher = better')
print('=' * 68)
print(f'  {"Metric":<34} {"Before":>10} {"After":>10} {"Delta":>8}')
print('  ' + '-' * 66)
print(f'  {"BARTScore (bidir mean)":<34} {b_bart_mean:>10.4f} {a_bart_mean:>10.4f} {a_bart_mean-b_bart_mean:>+8.4f}')
print(f'  {"BARTScore (bidir std)":<34} {b_bart_std:>10.4f} {a_bart_std:>10.4f}')
print('=' * 68)
print(f'  BARTScore: {"improved" if a_bart_mean > b_bart_mean else "did not improve"} ({a_bart_mean-b_bart_mean:+.4f})')

Loading facebook/bart-large (~1.6 GB on first run)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/1.02G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.02G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/513 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


BART loaded on cuda

Bidirectional BARTScore — BEFORE explanations...


Bidirectional BARTScore — AFTER explanations...


  BARTSCORE (bidirectional, facebook/bart-large)
  log [(P(exp|ref) + P(ref|exp)) / 2] per token — higher = better
  Metric                                 Before      After    Delta
  ------------------------------------------------------------------
  BARTScore (bidir mean)               -13.9733   -16.2259  -2.2526
  BARTScore (bidir std)                  1.4130     1.6347
  BARTScore: did not improve (-2.2526)


---
## Final Summary

In [11]:
print('=' * 72)
print('  COUNTERFACTUAL EVALUATION — BEFORE vs AFTER FINE-TUNING')
print('=' * 72)
print(f'  {"Metric":<34} {"Before":>10} {"After":>10} {"Delta":>8}')
print('  ' + '-' * 66)
print(f'  {"Avg explanation length (words)":<34} {df["before_len"].mean():>10.1f} {df["after_len"].mean():>10.1f}')
print(f'  {"Template fallback rate":<34} {df["before_template"].mean()*100:>9.1f}% {df["after_template"].mean()*100:>9.1f}%')
print(f'  {"BMR (Bridge Mention Rate)":<34} {b_bmr:>10.4f} {a_bmr:>10.4f} {a_bmr-b_bmr:>+8.4f}')
print(f'  {"USR (Unique Sentence Rate)":<34} {b_usr:>10.4f} {a_usr:>10.4f} {a_usr-b_usr:>+8.4f}')
print(f'  {"Alpha-Sentiment Pearson r":<34} {b_r:>10.4f} {a_r:>10.4f} {a_r-b_r:>+8.4f}')
print(f'  {"Factual Precision":<34} {b_fp:>10.4f} {a_fp:>10.4f} {a_fp-b_fp:>+8.4f}')
print(f'  {"BARTScore":<34} {b_bart_mean:>10.4f} {a_bart_mean:>10.4f} {a_bart_mean-b_bart_mean:>+8.4f}')
print('=' * 72)
print()

verdicts = [
    ('USR',               a_usr > b_usr,                    a_usr - b_usr),
    ('BMR',               a_bmr > b_bmr,                    a_bmr - b_bmr),
    ('Alpha-Sent r',      abs(a_r) > abs(b_r),              a_r   - b_r),
    ('Factual Precision', a_fp  > b_fp,                     a_fp  - b_fp),
    ('BARTScore',         a_bart_mean > b_bart_mean,         a_bart_mean - b_bart_mean),
]
for name, improved, delta in verdicts:
    status = ' improved' if improved else ' did not improve'
    print(f'  {name:<22}: {status} ({delta:+.4f})')

  COUNTERFACTUAL EVALUATION — BEFORE vs AFTER FINE-TUNING
  Metric                                 Before      After    Delta
  ------------------------------------------------------------------
  Avg explanation length (words)           33.4      155.6
  Template fallback rate                   0.0%       0.0%
  BMR (Bridge Mention Rate)              0.4402     0.9332  +0.4930
  USR (Unique Sentence Rate)             0.9845     1.0000  +0.0155
  Alpha-Sentiment Pearson r             -0.0015     0.0118  +0.0133
  Factual Precision                      0.0832     0.4958  +0.4126
  BARTScore                            -13.9733   -16.2259  -2.2526

  USR                   :  improved (+0.0155)
  BMR                   :  improved (+0.4930)
  Alpha-Sent r          :  improved (+0.0133)
  Factual Precision     :  improved (+0.4126)
  BARTScore             :  did not improve (-2.2526)
